# Desafio Wellbe — ETL

Extração, tratamento e carga de `data/dados.csv` em MySQL.

In [1]:
import re
import pandas as pd

RAW_PATH = "../data/dados.csv"

raw = pd.read_csv(RAW_PATH, skiprows=13, encoding="utf-8")

colunas_esperadas = {
    "Código": "codigo",
    "Custo do afastamento": "custo_raw",
    "Identificação": "identificacao_raw",
    "Funcionário": "funcionario_raw",
    "Departamento": "departamento_raw",
    "Data do Atestado": "data_raw",
    "Especialidade": "especialidade_raw",
    "Motivo": "motivo_raw",
    "Líder": "lider_raw",
}
assert set(colunas_esperadas.keys()) <= set(raw.columns), (
    f"Header inesperado: {raw.columns.tolist()}"
)
raw = raw.rename(columns=colunas_esperadas)

print(raw.shape)
raw.head()

(91, 9)


,codigo,custo_raw,identificacao_raw,funcionario_raw,departamento_raw,data_raw,especialidade_raw,motivo_raw,lider_raw
0,1036743,"20,4",NaN,Anonimo 101,Gerente,29/05/2019,Exames,Exames,Sim
1,1036742,23,NaN,Anonimo 1,ASSISTENTE DE IMPLANTACAO,23/05/2019,Neurologia pediátrica,Acompanhamento familiar,NaN
2,1036741,"22,8",NaN,Anonimo 1,ASSISTENTE DE IMPLANTACAO,17/05/2019,NaN,Consulta médica,NaN
3,1036740,–-,NaN,Anonimo 1,ASSISTENTE DE IMPLANTACAO,16/05/2019,Exames,Exames,NaN
4,1036739,22,NaN,Anonimo 1,ASSISTENTE DE IMPLANTACAO,06/05/2019,NaN,Consulta médica,NaN


In [2]:
def parse_custo(value):
    if pd.isna(value):
        return 0.0
    text = str(value).strip().replace(",", ".")
    if re.fullmatch(r"-?\d+(\.\d+)?", text):
        return float(text)
    return 0.0


def parse_data(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NaT
    return pd.to_datetime(str(value).strip(), format="%d/%m/%Y", errors="coerce")


# Sanidade das regras de tratamento (DADOS.md)
assert parse_custo("20,4") == 20.4
assert parse_custo("--") == 0.0
assert parse_custo("–-") == 0.0
assert parse_custo(float("nan")) == 0.0
assert parse_custo("9215") == 9215.0  # valor bruto quebrado, tratado literalmente
assert parse_data("29/05/2019") == pd.Timestamp("2019-05-29")
assert pd.isna(parse_data(""))
assert pd.isna(parse_data(float("nan")))
print("helpers ok")

helpers ok
